Convert Yak model.pt into a Hugging Face-compatible model
This notebook targets the classifier checkpoint from the feature/finetune branch of gowthambalachandhiran/Yak.

It preserves Yak's original inference design:
1. Encode the input text (or SNLI premise + hypothesis).
2. Append candidate label names to the same sequence.
3. Run the Yak encoder.
4. Gather the hidden state at the first token of each candidate label.
5. Apply the shared Linear(128, 1) scorer and softmax over candidates.
   The exported model is a Hugging Face PreTrainedModel with save_pretrained() / from_pretrained() support. It is not converted into a fixed-output AutoModelForSequenceClassification, because that would change Yak's candidate-label architecture.
Run this notebook from the Yak repository root, or set REPO_DIR below. The checkpoint is expected at runs/model.pt by default; edit CHECKPOINT_PATH if yours is elsewhere.

In [ ]:
# 1. Locate the repository and install dependencies if needed
from pathlib import Path
import os, sys, json, shutil, re, math

# EDIT THIS if the notebook is not launched from inside the Yak repository.
REPO_DIR = Path.cwd()

def find_repo(start: Path):
    for p in [start, *start.parents]:
        if (p / "Model" / "MlmModel.py").exists() and (p / "Tokenizer" / "Tokenizer.py").exists():
            return p
    raise FileNotFoundError(
        "Could not find the Yak repository. Set REPO_DIR to the folder containing Model/ and Tokenizer/."
    )

REPO_DIR = find_repo(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Yak repository:", REPO_DIR)


In [ ]:
# 1. Locate the repository and install dependencies if needed
from pathlib import Path
import os, sys, json, shutil, re, math

# EDIT THIS if the notebook is not launched from inside the Yak repository.
REPO_DIR = Path.cwd()

def find_repo(start: Path):
    for p in [start, *start.parents]:
        if (p / "Model" / "MlmModel.py").exists() and (p / "Tokenizer" / "Tokenizer.py").exists():
            return p
    raise FileNotFoundError(
        "Could not find the Yak repository. Set REPO_DIR to the folder containing Model/ and Tokenizer/."
    )

REPO_DIR = find_repo(REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Yak repository:", REPO_DIR)


In [3]:
# 2. Configure checkpoint, tokenizer, metadata, and export paths
CHECKPOINT_PATH = REPO_DIR / "runs" /"mixture" /"best.pt"
TOKENIZER_PATH = REPO_DIR / "Tokenizer" / "TokenFile" / "bpe.json"
LABEL_METADATA_PATH = REPO_DIR / "Data" / "processed" / "label_metadata.json"
EXPORT_DIR = REPO_DIR / "runs" / "huggingface_yak_classifier"

# If your checkpoint is stored elsewhere, replace CHECKPOINT_PATH, for example:
# CHECKPOINT_PATH = REPO_DIR / ".Yak" / "runs" / "model.pt"

# for name, path in {
#     "checkpoint": CHECKPOINT_PATH,
#     "tokenizer": TOKENIZER_PATH,
#     "label metadata": LABEL_METADATA_PATH,
# }.items():
#     print(f"{name:16s}: {path} | exists={path.exists()}")

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"
assert TOKENIZER_PATH.exists(), f"BPE tokenizer file not found: {TOKENIZER_PATH}"
assert LABEL_METADATA_PATH.exists(), f"Label metadata not found: {LABEL_METADATA_PATH}"


In [ ]:
# 3. Install/import Hugging Face dependencies
%pip install -q -U transformers safetensors

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

from Model.LayerOne.EmbedPostional import EmbedPos
from Model.EncoderBlock import EncoderBlock
from Tokenizer.Tokenizer import BPE
from Model.OurExperiments.FineTuning.Utils.FastBpe import FastBPE, normalize_label

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)


In [5]:
# 4. Load Yak's custom BPE tokenizer and inspect its special-token IDs
bpe = BPE()
bpe.load(str(TOKENIZER_PATH))
fast_bpe = FastBPE(bpe)

print("Vocabulary size:", fast_bpe.vocab_size)
print("Special tokens:", bpe.special_tokens)
print({
    "pad": fast_bpe.pad_id,
    "unk": fast_bpe.unk_id,
    "cls": fast_bpe.cls_id,
    "sep": fast_bpe.sep_id,
    "mask": fast_bpe.mask_id,
})

assert fast_bpe.vocab_size == 30005, (
    f"Expected the 30,005-token Yak vocabulary described by this branch; got {fast_bpe.vocab_size}. "
    "Check that TOKENIZER_PATH points to the matching tokenizer checkpoint."
)


Vocabulary size: 30005
Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
{'pad': 0, 'unk': 1, 'cls': 2, 'sep': 3, 'mask': 4}


Define the Hugging Face model wrapper
The parameter names deliberately match the repository's YakClassifier (embed, blocks, final_norm, scorer). This allows the classifier checkpoint to load without changing its learned weights.

In [6]:

import torch
import torch.nn as nn
from transformers import PretrainedConfig, PreTrainedModel


class YakHFConfig(PretrainedConfig):

    model_type = "yak-candidate-classifier"

    def __init__(
        self,
        vocab_size=30005,
        d_model=128,
        nhead=4,
        num_layers=4,
        max_seq_len=2048,
        dropout=0.1,
        pad_token_id=0,
        **kwargs,
    ):
        super().__init__(
            pad_token_id=pad_token_id,
            **kwargs
        )

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        # Keep aliases for compatibility with original Yak code
        self.layers = num_layers
        self.max_seq = max_seq_len
        self.pad_id = pad_token_id


class YakHFForCandidateClassification(PreTrainedModel):

    config_class = YakHFConfig
    base_model_prefix = "yak"

    # Yak does not use tied weights
    all_tied_weights_keys = {}

    def __init__(self, config):

        super().__init__(config)

        # Embedding and positional encoding
        self.embed = EmbedPos(
            config.vocab_size,
            config.d_model,
            config.max_seq_len,
            config.pad_token_id,
            config.dropout,
        )

        # Transformer encoder blocks
        self.blocks = nn.ModuleList([
            EncoderBlock(
                config.d_model,
                config.nhead,
                config.dropout,
            )
            for _ in range(config.num_layers)
        ])

        # Final normalization and shared candidate scorer
        self.final_norm = nn.LayerNorm(config.d_model)

        self.scorer = nn.Linear(config.d_model, 1)

        # Let Hugging Face initialize the model
        self.post_init()

        # --------------------------------------------------
        # IMPORTANT: Persist positional encoding in checkpoint
        # --------------------------------------------------

        # Make positional encoding part of state_dict()
        self.embed._non_persistent_buffers_set.discard("pos_emb")

        # Rebuild the exact sinusoidal positional encoding
        # used by the original Yak model.
        with torch.no_grad():

            expected_pos = EmbedPos._get_positional_encoding(
                config.max_seq_len,
                config.d_model,
            )

            self.embed.pos_emb.copy_(
                expected_pos.to(
                    device=self.embed.pos_emb.device,
                    dtype=self.embed.pos_emb.dtype,
                )
            )

    def forward(
        self,
        input_ids=None,
        label_positions=None,
        label_mask=None,
        attention_mask=None,
        **kwargs,
    ):

        if input_ids is None:
            raise ValueError("input_ids must be provided")

        if label_positions is None:
            raise ValueError("label_positions must be provided")

        if label_mask is None:
            raise ValueError("label_mask must be provided")

        # Identify non-padding tokens
        keep = input_ids != self.config.pad_token_id

        # Yak's original encoder uses True for valid tokens
        padding_mask = None if bool(keep.all()) else keep

        # Embedding
        x = self.embed(input_ids)

        # Transformer encoder blocks
        for block in self.blocks:
            x = block(x, padding_mask)

        # Final normalization
        x = self.final_norm(x)

        # Gather hidden states at candidate-label positions
        batch_size = x.size(0)

        batch_indices = torch.arange(
            batch_size,
            device=x.device
        ).unsqueeze(1)

        candidate_hidden = x[batch_indices, label_positions]

        # Shared candidate scorer
        scores = self.scorer(candidate_hidden).squeeze(-1)

        # Mask unused candidate positions
        scores = scores.masked_fill(
            ~label_mask.bool(),
            float("-inf")
        )

        return scores


# --------------------------------------------------
# Instantiate model and inspect parameter count
# --------------------------------------------------

config = YakHFConfig()

model = YakHFForCandidateClassification(config)

print("Model initialized successfully!")
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Vocabulary size:", config.vocab_size)
print("Number of layers:", config.num_layers)
print("Maximum sequence length:", config.max_seq_len)

# Verify positional encoding is persistent
print(
    "Positional encoding in state_dict:",
    "embed.pos_emb" in model.state_dict()
)

print(
    "Positional encoding shape:",
    tuple(model.embed.pos_emb.shape)
)


Model initialized successfully!
Model parameters: 4634113
Vocabulary size: 30005
Number of layers: 4
Maximum sequence length: 2048
Positional encoding in state_dict: True
Positional encoding shape: (1, 2048, 128)


Load model.pt
This loader accepts common checkpoint containers (state_dict, model_state_dict, model, etc.) and strips common wrappers such as module.. It then checks that the checkpoint contains the classifier scorer as well as the encoder. A phase-1 MLM-only checkpoint is not sufficient for this notebook.

In [7]:
def extract_state_dict(checkpoint_obj):
    obj = checkpoint_obj
    if not isinstance(obj, dict):
        raise TypeError(f"Expected a dict-like checkpoint, got {type(obj)}")

    for key in ("model_state_dict", "model_state", "state_dict", "model", "classifier"):
        if key in obj and isinstance(obj[key], dict):
            obj = obj[key]
            break

    if not isinstance(obj, dict) or not obj:
        raise ValueError("Could not find a non-empty state_dict in the checkpoint.")

    state = dict(obj)
    prefixes = ("module.", "_orig_mod.", "model.", "classifier.")
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if state and all(k.startswith(prefix) for k in state):
                state = {k[len(prefix):]: v for k, v in state.items()}
                changed = True
                break
    return state


try:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
except TypeError:
    raw_checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

state_dict = extract_state_dict(raw_checkpoint)
print("Checkpoint tensors extracted:", len(state_dict))


Checkpoint tensors extracted: 53


In [8]:

# --------------------------------------------------
# Validate and load original Yak classifier checkpoint
# --------------------------------------------------

expected_keys = set(model.state_dict().keys())

filtered_state = {
    k: v
    for k, v in state_dict.items()
    if k in expected_keys
}

missing = sorted(expected_keys - set(filtered_state))

unexpected_model_keys = sorted(
    k for k in state_dict
    if k.startswith(
        ("embed.", "blocks.", "final_norm.", "scorer.")
    )
    and k not in expected_keys
)

print("Checkpoint tensors:", len(state_dict))
print("Matching model tensors:", len(filtered_state))
print("Missing model tensors:", missing)
print("Unexpected Yak tensors:", unexpected_model_keys)

# Verify that this is a classifier checkpoint
if not any(k.startswith("scorer.") for k in filtered_state):
    raise RuntimeError(
        "No scorer.* weights were found. This appears to be an "
        "MLM/pretraining checkpoint, not the classifier checkpoint."
    )

# The original Yak checkpoint does not store positional encoding.
# Our Hugging Face model now stores it as a persistent buffer.
# Therefore, allow exactly this one missing key.
allowed_missing = {"embed.pos_emb"}

if set(missing) != allowed_missing:
    raise RuntimeError(
        "Unexpected missing checkpoint tensors. "
        f"Expected only {allowed_missing}, got {set(missing)}"
    )

if unexpected_model_keys:
    raise RuntimeError(
        "Unexpected Yak tensors found: "
        f"{unexpected_model_keys}"
    )

# Load all learned parameters.
# Keep the positional encoding generated in the HF model.
load_result = model.load_state_dict(
    filtered_state,
    strict=False
)

# Validate the actual loading result.
if set(load_result.missing_keys) != allowed_missing:
    raise RuntimeError(
        f"Unexpected missing keys after loading: "
        f"{load_result.missing_keys}"
    )

if load_result.unexpected_keys:
    raise RuntimeError(
        f"Unexpected keys after loading: "
        f"{load_result.unexpected_keys}"
    )

model.to(device).eval()

# Verify positional encoding remains intact.
assert "embed.pos_emb" in model.state_dict()

print("\nClassifier checkpoint loaded successfully.")
print("Missing keys (expected):", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print(
    "Positional encoding range:",
    model.embed.pos_emb.min().item(),
    model.embed.pos_emb.max().item()
)


Checkpoint tensors: 53
Matching model tensors: 53
Missing model tensors: ['embed.pos_emb']
Unexpected Yak tensors: []

Classifier checkpoint loaded successfully.
Missing keys (expected): ['embed.pos_emb']
Unexpected keys: []
Positional encoding range: -1.0 1.0


Export in Hugging Face save_pretrained() format
The export includes the model weights/configuration, the original Yak BPE merges, and the task label metadata. Keep the tokenizer and model together: the custom BPE vocabulary must match the embedding matrix.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(EXPORT_DIR, safe_serialization=True)
shutil.copy2(TOKENIZER_PATH, EXPORT_DIR / "bpe.json")
shutil.copy2(LABEL_METADATA_PATH, EXPORT_DIR / "label_metadata.json")
shutil.copy2(REPO_DIR / "yak_inference.py", EXPORT_DIR / "yak_inference.py")
shutil.copy2(REPO_DIR / "requirements.txt", EXPORT_DIR / "requirements.txt")
shutil.copy2(REPO_DIR / "Model" / "OurExperiments" / "HuggingFace" / "README.md", EXPORT_DIR / "README.md")

# A small manifest records the custom inference contract.
manifest = {
    "model_type": "yak-candidate-classifier",
    "architecture": "YakClassifier",
    "candidate_label_scoring": "shared Linear(d_model, 1) over first-token hidden states",
    "tokenizer": "Yak custom BPE; use bpe.json and yak_inference.py",
    "tasks": ["banking77", "clinc", "snli"],
    "max_seq_len": config.max_seq_len,
    "max_text_tokens": 96,
    "max_premise_tokens": 64,
    "max_hypothesis_tokens": 32,
    "max_label_tokens": 12,
}
(EXPORT_DIR / "yak_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Exported to:", EXPORT_DIR)
print("Files:", sorted(p.name for p in EXPORT_DIR.iterdir()))


8. Build candidate-label inputs exactly as in Yak
   For ordinary classification, the packed sequence is:
[CLS] text [SEP] label_1 [SEP] label_2 [SEP] ...

For SNLI, Yak joins the two sentences with a separator before appending candidate labels. The label order is preserved in the returned probability list.

In [10]:
LABEL_ALIASES = {"oos": "out of scope"}

def encode_text_for_task(text=None, premise=None, hypothesis=None,
                         max_text_tokens=96, max_premise_tokens=64, max_hypothesis_tokens=32):
    if premise is not None or hypothesis is not None:
        if premise is None or hypothesis is None:
            raise ValueError("Provide both premise and hypothesis for SNLI.")
        p_ids = fast_bpe.encode_ids(premise, normalize=True)[:max_premise_tokens]
        h_ids = fast_bpe.encode_ids(hypothesis, normalize=True)[:max_hypothesis_tokens]
        return p_ids + [fast_bpe.sep_id] + h_ids
    if text is None:
        raise ValueError("Provide text, or both premise and hypothesis.")
    return fast_bpe.encode_ids(text, normalize=True)[:max_text_tokens]


def pack_candidate_labels(text_ids, candidate_labels, max_label_tokens=12):
    if not candidate_labels:
        raise ValueError("candidate_labels must contain at least one label.")

    ids = [fast_bpe.cls_id] + list(text_ids) + [fast_bpe.sep_id]
    label_positions = []

    for label in candidate_labels:
        label_text = LABEL_ALIASES.get(label, normalize_label(label))
        label_ids = fast_bpe.encode_ids(label_text, normalize=True)[:max_label_tokens]
        if not label_ids:
            label_ids = [fast_bpe.unk_id]
        label_positions.append(len(ids))
        ids.extend(label_ids)
        ids.append(fast_bpe.sep_id)

    if len(ids) > config.max_seq_len:
        raise ValueError(
            f"Packed input has {len(ids)} tokens, exceeding max_seq_len={config.max_seq_len}. "
            "Reduce the text length or candidate-label count."
        )

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    label_positions = torch.tensor([label_positions], dtype=torch.long, device=device)
    label_mask = torch.ones_like(label_positions, dtype=torch.bool, device=device)
    return input_ids, label_positions, label_mask


@torch.inference_mode()
def predict_candidates(candidate_labels, text=None, premise=None, hypothesis=None):
    model.eval()
    text_ids = encode_text_for_task(text=text, premise=premise, hypothesis=hypothesis)
    input_ids, label_positions, label_mask = pack_candidate_labels(text_ids, candidate_labels)
    outputs = model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )
    
    probabilities = torch.softmax(outputs.float(), dim=-1)[0].cpu().tolist()
    ranked = sorted(
        [{"label": label, "score": float(score)} for label, score in zip(candidate_labels, probabilities)],
        key=lambda item: item["score"],
        reverse=True,
    )
    return ranked


Load the exported model back from disk
This verifies that the Hugging Face checkpoint can be reconstructed independently of the in-memory model used during conversion.

In [11]:
reloaded_config = YakHFConfig.from_pretrained(EXPORT_DIR)
reloaded_model = YakHFForCandidateClassification.from_pretrained(
    EXPORT_DIR,
    config=reloaded_config,
).to(device).eval()

print("Reloaded model:", type(reloaded_model).__name__)
print("Reloaded parameters:", sum(p.numel() for p in reloaded_model.parameters()))


Loading weights: 100%|██████████| 54/54 [00:00<00:00, 3036.62it/s]

Reloaded model: YakHFForCandidateClassification
Reloaded parameters: 4634113


Run inference
The metadata file contains the label vocabularies for all three tasks. Select the relevant task and pass its candidate labels. For SNLI, pass both premise and hypothesis.

In [12]:
label_metadata = json.loads(LABEL_METADATA_PATH.read_text(encoding="utf-8"))
label_vocabularies = label_metadata["label_vocabularies"]

print("Available tasks:", list(label_vocabularies))
for task_name, names in label_vocabularies.items():
    print(f"{task_name:10s}: {len(names)} candidate labels")

# Example: SNLI inference
snli_labels = label_vocabularies["snli"]
snli_result = predict_candidates(
    candidate_labels=snli_labels,
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)
snli_result[:6]


Available tasks: ['snli', 'banking77', 'clinc']
snli      : 3 candidate labels
banking77 : 77 candidate labels
clinc     : 151 candidate labels


[{'label': 'entailment', 'score': 0.9278973937034607},
 {'label': 'contradiction', 'score': 0.039467014372348785},
 {'label': 'neutral', 'score': 0.03263559937477112}]

Check numerical equivalence against the original loaded model
The exported and reloaded model should produce the same logits as the in-memory model for an identical packed input. This is a useful conversion sanity check; it does not measure task accurac

In [13]:

import math
import torch

model.eval()
reloaded_model.eval()

# 1. Compare runtime buffers, including positional embeddings.
original_buffers = dict(model.named_buffers())
reloaded_buffers = dict(reloaded_model.named_buffers())

print("Original buffers:", original_buffers.keys())
print("Reloaded buffers:", reloaded_buffers.keys())

for name in original_buffers:
    a = original_buffers[name].detach().cpu()
    b = reloaded_buffers[name].detach().cpu()

    print(
        name,
        "shape:", tuple(a.shape),
        "max difference:", (a.float() - b.float()).abs().max().item()
    )

# 2. Compare token embeddings and positional embeddings separately.
text_ids = encode_text_for_task(
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

input_ids, label_positions, label_mask = pack_candidate_labels(
    text_ids, snli_labels
)

with torch.inference_mode():
    original_tok = (
        model.embed.tok_emb(input_ids)
        * math.sqrt(model.config.d_model)
    )
    reloaded_tok = (
        reloaded_model.embed.tok_emb(input_ids)
        * math.sqrt(reloaded_model.config.d_model)
    )

    seq_len = input_ids.size(1)

    original_pos = model.embed.pos_emb[:, :seq_len, :]
    reloaded_pos = reloaded_model.embed.pos_emb[:, :seq_len, :]

    original_emb = model.embed(input_ids)
    reloaded_emb = reloaded_model.embed(input_ids)

print("\nToken embedding difference:",
      (original_tok - reloaded_tok).abs().max().item())

print("Positional embedding difference:",
      (original_pos - reloaded_pos).abs().max().item())

print("Embedding module training states:",
      model.embed.training,
      reloaded_model.embed.training)

print("Complete embedding difference:",
      (original_emb - reloaded_emb).abs().max().item())


Original buffers: dict_keys(['embed.pos_emb'])
Reloaded buffers: dict_keys(['embed.pos_emb'])
embed.pos_emb shape: (1, 2048, 128) max difference: 0.0

Token embedding difference: 0.0
Positional embedding difference: 0.0
Embedding module training states: False False
Complete embedding difference: 0.0


In [14]:

print("Original positional encoding:")
print(model.embed.pos_emb[0, :4, :8])

print("\nReloaded positional encoding:")
print(reloaded_model.embed.pos_emb[0, :4, :8])

print("\nOriginal min/max:",
      model.embed.pos_emb.min().item(),
      model.embed.pos_emb.max().item())

print("Reloaded min/max:",
      reloaded_model.embed.pos_emb.min().item(),
      reloaded_model.embed.pos_emb.max().item())

# Independently regenerate the expected positional encoding.
expected = EmbedPos._get_positional_encoding(2048, 128).to(device)

print("\nOriginal vs expected:",
      (model.embed.pos_emb - expected).abs().max().item())

print("Reloaded vs expected:",
      (reloaded_model.embed.pos_emb - expected).abs().max().item())


Original positional encoding:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.7617,  0.6479,  0.6816,  0.7318,  0.6047,  0.7965],
        [ 0.9093, -0.4161,  0.9870, -0.1604,  0.9975,  0.0709,  0.9632,  0.2687],
        [ 0.1411, -0.9900,  0.5173, -0.8558,  0.7783, -0.6279,  0.9296, -0.3685]])

Reloaded positional encoding:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.7617,  0.6479,  0.6816,  0.7318,  0.6047,  0.7965],
        [ 0.9093, -0.4161,  0.9870, -0.1604,  0.9975,  0.0709,  0.9632,  0.2687],
        [ 0.1411, -0.9900,  0.5173, -0.8558,  0.7783, -0.6279,  0.9296, -0.3685]])

Original min/max: -1.0 1.0
Reloaded min/max: -1.0 1.0

Original vs expected: 0.0
Reloaded vs expected: 0.0


In [15]:
@torch.inference_mode()
def compare_intermediate_outputs(input_ids, label_positions):

    def get_intermediates(m):
        x = m.embed(input_ids)

        outputs = {"embedding": x.detach().clone()}

        keep = input_ids != m.config.pad_token_id
        padding_mask = None if bool(keep.all()) else keep

        for i, block in enumerate(m.blocks):
            x = block(x, padding_mask)
            outputs[f"block_{i}"] = x.detach().clone()

        x = m.final_norm(x)
        outputs["final_norm"] = x.detach().clone()

        batch_indices = torch.arange(
            x.size(0), device=x.device
        ).unsqueeze(1)

        candidate_hidden = x[batch_indices, label_positions]
        outputs["candidate_hidden"] = candidate_hidden.detach().clone()

        outputs["scores"] = m.scorer(candidate_hidden).squeeze(-1)

        return outputs

    original = get_intermediates(model)
    reloaded = get_intermediates(reloaded_model)

    for name in original:
        diff = (
            original[name].float() - reloaded[name].float()
        ).abs().max().item()

        print(f"{name:25s}: {diff:.10f}")


text_ids = encode_text_for_task(
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

input_ids, label_positions, label_mask = pack_candidate_labels(
    text_ids, snli_labels
)

compare_intermediate_outputs(input_ids, label_positions)

embedding                : 0.0000000000
block_0                  : 0.0000000000
block_1                  : 0.0000000000
block_2                  : 0.0000000000
block_3                  : 0.0000000000
final_norm               : 0.0000000000
candidate_hidden         : 0.0000000000
scores                   : 0.0000000000


In [16]:

with torch.no_grad():
    reloaded_model.embed.pos_emb.copy_(
        model.embed.pos_emb
    )

print(
    "Positional encoding difference:",
    (model.embed.pos_emb - reloaded_model.embed.pos_emb)
    .abs().max().item()
)

compare_intermediate_outputs(input_ids, label_positions)


Positional encoding difference: 0.0
embedding                : 0.0000000000
block_0                  : 0.0000000000
block_1                  : 0.0000000000
block_2                  : 0.0000000000
block_3                  : 0.0000000000
final_norm               : 0.0000000000
candidate_hidden         : 0.0000000000
scores                   : 0.0000000000


In [17]:

import torch

model.eval()
reloaded_model.eval()

# 1. Prepare a manual SNLI example
text_ids = encode_text_for_task(
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

# 2. Pack candidate labels
input_ids, label_positions, label_mask = pack_candidate_labels(
    text_ids,
    snli_labels,
)

# 3. Move inputs to the appropriate device
input_ids = input_ids.to(device)
label_positions = label_positions.to(device)
label_mask = label_mask.to(device)

# 4. Compare the inputs
print("Input IDs shape:", input_ids.shape)
print("Label positions shape:", label_positions.shape)
print("Label mask shape:", label_mask.shape)

# 5. Compare intermediate outputs
@torch.inference_mode()
def get_intermediates(m):
    x = m.embed(input_ids)

    outputs = {"embedding": x.detach().clone()}

    keep = input_ids != m.config.pad_token_id
    padding_mask = None if bool(keep.all()) else keep

    for i, block in enumerate(m.blocks):
        x = block(x, padding_mask)
        outputs[f"block_{i}"] = x.detach().clone()

    x = m.final_norm(x)
    outputs["final_norm"] = x.detach().clone()

    batch_indices = torch.arange(
        x.size(0), device=x.device
    ).unsqueeze(1)

    candidate_hidden = x[batch_indices, label_positions]

    outputs["candidate_hidden"] = candidate_hidden.detach().clone()

    outputs["scores"] = m.scorer(candidate_hidden).squeeze(-1)

    return outputs


original_outputs = get_intermediates(model)
reloaded_outputs = get_intermediates(reloaded_model)

print("\nIntermediate output comparison:")

for name in original_outputs:
    diff = (
        original_outputs[name].float()
        - reloaded_outputs[name].float()
    ).abs().max().item()

    print(f"{name:25s}: {diff:.10f}")


Input IDs shape: torch.Size([1, 28])
Label positions shape: torch.Size([1, 3])
Label mask shape: torch.Size([1, 3])

Intermediate output comparison:
embedding                : 0.0000000000
block_0                  : 0.0000000000
block_1                  : 0.0000000000
block_2                  : 0.0000000000
block_3                  : 0.0000000000
final_norm               : 0.0000000000
candidate_hidden         : 0.0000000000
scores                   : 0.0000000000


In [18]:

@torch.inference_mode()
def compare_original_and_reloaded(
    candidate_labels,
    text=None,
    premise=None,
    hypothesis=None,
):
    text_ids = encode_text_for_task(
        text=text,
        premise=premise,
        hypothesis=hypothesis,
    )

    input_ids, label_positions, label_mask = pack_candidate_labels(
        text_ids,
        candidate_labels,
    )

    # Original model
    original_logits = model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )

    # Reloaded Hugging Face model
    reloaded_logits = reloaded_model(
        input_ids=input_ids,
        label_positions=label_positions,
        label_mask=label_mask,
    )

    # Compare raw candidate scores
    max_abs_diff = (
        original_logits - reloaded_logits
    ).abs().max().item()

    print("Maximum absolute logit difference:", max_abs_diff)

    assert torch.allclose(
        original_logits,
        reloaded_logits,
        atol=1e-6,
        rtol=1e-5,
    ), "Mismatch between original and reloaded model!"

    print("PASS: Exported model reproduces the original model scores.")


compare_original_and_reloaded(
    candidate_labels=snli_labels,
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)


Maximum absolute logit difference: 0.0
PASS: Exported model reproduces the original model scores.


In [19]:
print(
    "Positional encoding in state_dict:",
    "embed.pos_emb" in model.state_dict()
)

print(
    "Positional encoding range:",
    model.embed.pos_emb.min().item(),
    model.embed.pos_emb.max().item()
)

print("Checkpoint loading validation completed.")

Positional encoding in state_dict: True
Positional encoding range: -1.0 1.0
Checkpoint loading validation completed.


In [20]:
# Compare original model with Hugging Face reloaded model

with torch.no_grad():

    positional_difference = torch.max(
        torch.abs(
            model.embed.pos_emb -
            reloaded_model.embed.pos_emb
        )
    ).item()

print(
    "Positional encoding difference:",
    positional_difference
)

assert positional_difference == 0.0, (
    "Positional encoding mismatch after reloading!"
)

print("Positional encoding validation passed.")

Positional encoding difference: 0.0
Positional encoding validation passed.


In [21]:
original_state = model.state_dict()
reloaded_state = reloaded_model.state_dict()

original_keys = set(original_state.keys())
reloaded_keys = set(reloaded_state.keys())

missing_in_reloaded = sorted(original_keys - reloaded_keys)
extra_in_reloaded = sorted(reloaded_keys - original_keys)

print("Original model tensors:", len(original_keys))
print("Reloaded model tensors:", len(reloaded_keys))

print("\nMissing in reloaded model:")
for key in missing_in_reloaded:
    print("  ", key)

print("\nExtra in reloaded model:")
for key in extra_in_reloaded:
    print("  ", key)

Original model tensors: 54
Reloaded model tensors: 54

Missing in reloaded model:

Extra in reloaded model:


In [22]:
common_keys = original_keys.intersection(reloaded_keys)

mismatched_tensors = []

for key in sorted(common_keys):
    original_tensor = original_state[key].cpu()
    reloaded_tensor = reloaded_state[key].cpu()

    if original_tensor.shape != reloaded_tensor.shape:
        mismatched_tensors.append(
            (key, "Shape mismatch",
             original_tensor.shape, reloaded_tensor.shape)
        )
        continue

    max_diff = (
        original_tensor - reloaded_tensor
    ).abs().max().item()

    if max_diff != 0:
        mismatched_tensors.append(
            (key, "Value mismatch", max_diff)
        )

print("\nCommon tensors:", len(common_keys))
print("Mismatched tensors:", len(mismatched_tensors))

for item in mismatched_tensors:
    print(item)

if not mismatched_tensors:
    print("PASS: All common tensors are identical.")


Common tensors: 54
Mismatched tensors: 0
PASS: All common tensors are identical.


In [23]:
compare_original_and_reloaded(
    candidate_labels=snli_labels,
    premise="A person is riding a bicycle down the street.",
    hypothesis="Someone is outdoors.",
)

Maximum absolute logit difference: 0.0
PASS: Exported model reproduces the original model scores.


In [24]:
print("Original model:")
print("pos_emb in state_dict:", "embed.pos_emb" in model.state_dict())
print(
    "Non-persistent buffers:",
    model.embed._non_persistent_buffers_set
)

print("\nReloaded model:")
print(
    "pos_emb in state_dict:",
    "embed.pos_emb" in reloaded_model.state_dict()
)
print(
    "Non-persistent buffers:",
    reloaded_model.embed._non_persistent_buffers_set
)

Original model:
pos_emb in state_dict: True
Non-persistent buffers: set()

Reloaded model:
pos_emb in state_dict: True
Non-persistent buffers: set()


In [25]:
from safetensors.torch import load_file

weights_path = EXPORT_DIR / "model.safetensors"

saved_weights = load_file(str(weights_path))

print(
    "embed.pos_emb in saved file:",
    "embed.pos_emb" in saved_weights
)

print("Saved tensor count:", len(saved_weights))

embed.pos_emb in saved file: True
Saved tensor count: 54


12. Publish the self-contained Python model package to Hugging Face
Run this only after the custom runtime modules have been added to EXPORT_DIR and tested. The Hub upload must include the custom Yak architecture and inference helper, not just weights and tokenizer metadata. This cell creates a public model repository and uploads the folder.

In [ ]:
# Public Hugging Face publication
%pip install -q -U huggingface_hub

from huggingface_hub import HfApi, login

HF_REPO_ID = "gbalachandhiran/yak-candidate-classifier"

# These files are required for independent Python consumption.
# Do not upload a weights-only export: consumers need the exact Yak runtime.
required_runtime_files = ["yak_inference.py"]
missing_runtime_files = [
    name for name in required_runtime_files
    if not (EXPORT_DIR / name).is_file()
]
if missing_runtime_files:
    raise RuntimeError(
        "Refusing to publish an incomplete model package. Add and test these files first: "
        + ", ".join(missing_runtime_files)
    )

required_export_files = [
    "config.json",
    "model.safetensors",
    "bpe.json",
    "label_metadata.json",
    "yak_manifest.json",
    "yak_inference.py",
    "requirements.txt",
    "README.md",
]
missing_export_files = [
    name for name in required_export_files
    if not (EXPORT_DIR / name).is_file()
]
if missing_export_files:
    raise RuntimeError(
        "Export is incomplete. Missing: " + ", ".join(missing_export_files)
    )

readme_path = EXPORT_DIR / "README.md"
shutil.copy2(REPO_DIR / "Model" / "OurExperiments" / "HuggingFace" / "README.md", readme_path)

# Login prompts securely for a Hugging Face access token; do not hard-code tokens.
login()

api = HfApi()
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)

commit_info = api.upload_folder(
    folder_path=str(EXPORT_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Publish Yak candidate classifier and Python inference package",
    ignore_patterns=["__pycache__/**", "*.pyc"],
    delete_patterns=["__pycache__/**", "*.pyc"],
)

print("Published model:", f"https://huggingface.co/{HF_REPO_ID}")
print("Commit:", getattr(commit_info, "oid", "uploaded"))


In [28]:
candidate_labels = [
    "transfer money",
    "check account balance",
    "report a lost card",
    "cancel a transaction",
]

results = predict_candidates(
    candidate_labels,
    text="I need to transfer money to another account",
)

for result in results:
    print(f"{result['label']}: {result['score']:.4f}")

transfer money: 0.5687
check account balance: 0.2210
report a lost card: 0.1413
cancel a transaction: 0.0689


## Local Testing - Pan ultimate test

In [30]:
import sys
from pathlib import Path

# Make the locally exported inference module importable
export_dir = Path(EXPORT_DIR)
if str(export_dir) not in sys.path:
    sys.path.insert(0, str(export_dir))

from yak_inference import YakClassifier

classifier = YakClassifier.from_pretrained(export_dir, device=device)

candidate_labels = [
    "transfer money",
    "check account balance",
    "report a lost card",
    "cancel a transaction",
]

results = classifier.predict(
    "I need to transfer money to another account",
    candidate_labels=candidate_labels,
)

for result in results:
    print(f"{result['label']}: {result['score']:.4f}")

assert abs(sum(result["score"] for result in results) - 1.0) < 1e-5

Loading weights: 100%|██████████| 54/54 [00:00<00:00, 2264.56it/s]


transfer money: 0.7011
report a lost card: 0.1201
check account balance: 0.1020
cancel a transaction: 0.0768


In [31]:
from pathlib import Path
import importlib.util
import sys
import torch
from huggingface_hub import HfApi, login, snapshot_download

HF_REPO_ID = "gbalachandhiran/yak-candidate-classifier"
export_dir = Path(EXPORT_DIR).resolve()

required_files = [
    "config.json",
    "model.safetensors",
    "bpe.json",
    "label_metadata.json",
    "yak_manifest.json",
    "yak_inference.py",
    "requirements.txt",
]
missing = [name for name in required_files if not (export_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Export is missing required files: {missing}")

# Authenticate, create a public model repository if needed, and upload the package.
login()
api = HfApi()
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=False,
    exist_ok=True,
)
commit = api.upload_folder(
    folder_path=str(export_dir),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Publish Yak candidate classifier and inference API",
)
print("Uploaded commit:", commit.oid)

# Download that exact commit into a separate local folder.
download_dir = export_dir.parent / "huggingface_hub_download_test"
download_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="model",
    revision=commit.oid,
    local_dir=str(download_dir),
    force_download=True,
)
print("Downloaded to:", download_dir)

# Import the inference code from the downloaded folder (not the original export).
spec = importlib.util.spec_from_file_location(
    "yak_inference_hub_download_test",
    download_dir / "yak_inference.py",
)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load yak_inference.py from {download_dir}")

inference_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = inference_module
spec.loader.exec_module(inference_module)

classifier = inference_module.YakClassifier.from_pretrained(
    download_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

candidate_labels = [
    "transfer money",
    "check account balance",
    "report a lost card",
    "cancel a transaction",
]
results = classifier.predict(
    "I need to transfer money to another account",
    candidate_labels=candidate_labels,
)

for result in results:
    print(f"{result['label']}: {result['score']:.4f}")

assert {result["label"] for result in results} == set(candidate_labels)
assert abs(sum(result["score"] for result in results) - 1.0) < 1e-5
print("Downloaded model test passed.")

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)' thrown while requesting POST https://huggingface.co/api/models/gbalachandhiran/yak-candidate-classifier/commit/main
Retrying in 1s [Retry 1/5].


Uploaded commit: f299e27b7838d813711a3e68dc1a2ba5e1b8d174


Fetching 9 files:   0%|          | 0/9 [00:03<?, ?it/s]


ValueError: Force download failed due to the above error.

In [33]:
import os
from pathlib import Path
import certifi

proxy_ca = Path.home() / "Downloads" / "company-proxy-root.pem"
if not proxy_ca.is_file():
    raise FileNotFoundError(proxy_ca)

bundle = Path(EXPORT_DIR).resolve().parent / "huggingface-ca-bundle.pem"
bundle.write_bytes(Path(certifi.where()).read_bytes() + b"\n" + proxy_ca.read_bytes())

os.environ["SSL_CERT_FILE"] = str(bundle)
os.environ["REQUESTS_CA_BUNDLE"] = str(bundle)
os.environ["CURL_CA_BUNDLE"] = str(bundle)

from huggingface_hub import snapshot_download

In [34]:
# Use Windows' trusted certificate store for HTTPS connections.
import truststore
truststore.inject_into_ssl()

from pathlib import Path
import importlib.util
import sys
import torch
from huggingface_hub import snapshot_download

download_dir = Path(EXPORT_DIR).resolve().parent / "huggingface_hub_download_test"
download_dir.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="model",
    revision=commit.oid,  # Downloads the exact commit you uploaded
    local_dir=str(download_dir),
    force_download=True,
)

spec = importlib.util.spec_from_file_location(
    "yak_inference_hub_download_test",
    download_dir / "yak_inference.py",
)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load yak_inference.py from {download_dir}")

inference_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = inference_module
spec.loader.exec_module(inference_module)

classifier = inference_module.YakClassifier.from_pretrained(
    download_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

candidate_labels = [
    "transfer money",
    "check account balance",
    "report a lost card",
    "cancel a transaction",
]
results = classifier.predict(
    "I need to transfer money to another account",
    candidate_labels=candidate_labels,
)

for result in results:
    print(f"{result['label']}: {result['score']:.4f}")

assert abs(sum(result["score"] for result in results) - 1.0) < 1e-5
print("Downloaded model test passed.")

Fetching 9 files:   0%|          | 0/9 [00:02<?, ?it/s]


ValueError: Force download failed due to the above error.

In [35]:
from pathlib import Path
from huggingface_hub import HfApi, login

export_dir = Path(EXPORT_DIR).resolve()
required = ["config.json", "model.safetensors", "bpe.json", "yak_inference.py"]
missing = [name for name in required if not (export_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Export folder is missing: {missing}")

login()

api = HfApi()
commit = api.upload_folder(
    folder_path=str(export_dir),
    repo_id="gbalachandhiran/yak-candidate-classifier",
    repo_type="model",
    commit_message="Upload Yak candidate classifier",
)
print("Uploaded commit:", commit.oid)

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1016)' thrown while requesting POST https://huggingface.co/api/models/gbalachandhiran/yak-candidate-classifier/preupload/main
Retrying in 1s [Retry 1/5].
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded commit: f299e27b7838d813711a3e68dc1a2ba5e1b8d174


In [36]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=str(export_dir),
    repo_id="gbalachandhiran/yak-candidate-classifier",
    repo_type="model",
    delete_patterns=["__pycache__/**", "*.pyc"],
    ignore_patterns=["__pycache__/**", "*.pyc"],
    commit_message="Remove Python cache files",
)

CommitInfo(commit_url='https://huggingface.co/gbalachandhiran/yak-candidate-classifier/commit/1115db81bb76229a751228c73add39d02911753c', commit_message='Remove Python cache files', commit_description='', oid='1115db81bb76229a751228c73add39d02911753c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gbalachandhiran/yak-candidate-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='gbalachandhiran/yak-candidate-classifier'), pr_revision=None, pr_num=None)

##Publishing card

In [ ]:
!pip -q install huggingface_hub transformers safetensors numpy

import sys
import torch
from huggingface_hub import snapshot_download

repo_id = "gbalachandhiran/yak-candidate-classifier"
model_dir = snapshot_download(repo_id=repo_id)
sys.path.insert(0, model_dir)

from yak_inference import YakClassifier

classifier = YakClassifier.from_pretrained(
    model_dir,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

labels = [
    "transfer money",
    "check account balance",
    "report a lost card",
    "cancel a transaction",
]
results = classifier.predict(
    "I need to transfer money to another account",
    candidate_labels=labels,
)

for item in results:
    print(f"{item['label']}: {item['score']:.4f}")

assert {item["label"] for item in results} == set(labels)
assert abs(sum(item["score"] for item in results) - 1.0) < 1e-5
print("Hub download and inference test passed.")

In [38]:
from pathlib import Path
from huggingface_hub import HfApi, login

card_path = Path(EXPORT_DIR) / "README.md"
if not card_path.is_file():
    raise FileNotFoundError(f"README.md not found: {card_path}")

login()
commit = HfApi().upload_file(
    path_or_fileobj=str(card_path),
    path_in_repo="README.md",
    repo_id="gbalachandhiran/yak-candidate-classifier",
    repo_type="model",
    commit_message="Add model card and inference instructions",
)
print(commit.oid)

58ebcda2bd43fa0bd181422904400763deb7601f


In [39]:
model

YakHFForCandidateClassification(
  (embed): EmbedPos(
    (dropout): Dropout(p=0.1, inplace=False)
    (tok_emb): Embedding(30005, 128, padding_idx=0)
  )
  (blocks): ModuleList(
    (0-3): 4 x EncoderBlock(
      (attention): BiDirectionalAttention(
        (qkv_proj): Linear(in_features=128, out_features=384, bias=True)
        (out_proj): Linear(in_features=128, out_features=128, bias=True)
      )
      (ffn): FFN(
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (activation): GELU(approximate='none')
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (scorer): Linear(in_features=128, out_features=1, bias=True)
)